<a href="https://colab.research.google.com/github/ulumbagas/CNN_code/blob/main/Transfer%20learning/transfer%20learning%20and%20pytorch%20experiment%20tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download Dataset from KAGGLE

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snmahsa/animal-image-dataset-cats-dogs-and-foxes")

print("Path to dataset files:", path)

'''
url: https://www.kaggle.com/datasets/snmahsa/animal-image-dataset-cats-dogs-and-foxes
'''

In [ ]:
"""
This script is used to explore and summarize dataset.
It scans through the dataset folder, counts the number of images in each class folder,
and organizes the results into a pandas DataFrame for easier analysis.
"""

import pandas as pd
import numpy as np
from pathlib import Path

# Define the dataset path (make sure 'path' variable is set correctly before running)
path = Path(path)

# Define the directory that contains all image data
data_dir=path/'Animal Image Dataset-Cats, Dogs, and Foxes'



# Print the total number of classes (subfolders) inside 'hiragana'
print(f'number of class: {len(list(data_dir.iterdir()))}')


img = []
img_files = []
img_directory=[]
# Ensure hiragana_dir is a Path object

for folder_path in data_dir.iterdir():
    if folder_path.is_dir():   # only process directories (ignore stray files if any)
        num_files = len(list(folder_path.iterdir()))
        img_directory.append(folder_path)
        img.append(folder_path.name)
        img_files.append(num_files)

# Create a pandas DataFrame to summarize the dataset
df = pd.DataFrame({'class': img,'Image Dir':img_directory ,'number of files': img_files})
df


## Plot data

In [ ]:
"""
This script is used to visualize random images from the Hiragana dataset.
It selects random class folders (labels), picks random images inside them,
and plots them in a grid layout using matplotlib.

"""
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import random
import torch
import torchvision
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchsummary import summary


# Get all class folders from dataset directory (each folder = one class)
img_dir=list(data_dir.iterdir())
kolom=5
baris=3

# Create a matplotlib figure with specified grid layout
fig, axes = plt.subplots(baris, kolom, figsize=(9, 4))  # 3 baris, 5 kolom
axes = axes.flatten()

# Loop through each subplot (axis) and display a random image
for _, ax in enumerate(axes):

    # Pick a random class folder
    label = random.choice(img_dir)

    # Pick a random image from that class folder
    images = list(label.iterdir())
    img_path = random.choice(images)

    # Read image using matplotlib
    animal_image = mpimg.imread(img_path)

    # Show the image on the subplot
    ax.imshow(animal_image)
    ax.axis("off")
    ax.set_title(f'class : {label.name}', fontsize=12)
plt.tight_layout()
plt.show()

# Data Augmentation

In [ ]:
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode
from torch import nn

train_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

## Split data

In [ ]:
try:
    import splitfolders
except ImportError:
    !pip install split-folders
    import splitfolders


splitfolders.ratio(
    input=data_dir,
    output='imgs',
    ratio=(0.7, 0.3)  # 70% train, 30% test
)

In [ ]:
img_path=Path('/content/imgs')
train_dir=img_path/'train'
val_dir=img_path/'val'

# Use ImageFolder to create dataset(s)
from torchvision import datasets
train_data = datasets.ImageFolder(root=train_dir, # target folder of images
                                  transform=train_transform, # transforms to perform on data (images)
                                  target_transform=None)
test_data = datasets.ImageFolder(root=val_dir,
                                 transform=test_transform,
                                 target_transform=None)

print(f"Train data:\n{train_data}\nTest data:\n{test_data}")

In [ ]:
bs=32
# Turn train and test Datasets into DataLoaders
train_loader = DataLoader(dataset=train_data,
                              batch_size=bs, # how many samples per batch?
                              num_workers=0, # how many subprocesses to use for data loading? (higher = more)
                              shuffle=True) # shuffle the data?

test_loader = DataLoader(dataset=test_data,
                             batch_size=bs,
                             num_workers=0,
                             shuffle=False) # don't usually need to shuffle testing data

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

# MODEL

In [ ]:
all_models = torchvision.models.list_models()
classification_models = torchvision.models.list_models(module=torchvision.models)
print(classification_models)

## ResNet18

In [ ]:
weights_ResNet18 = models.ResNet18_Weights.DEFAULT
resnet18 = models.resnet18(weights=weights_ResNet18)

#transform recomendation
preprocess = weights_ResNet18.transforms()
print(preprocess)


## ResNet50

In [ ]:
weights_ResNet50 = models.ResNet50_Weights.DEFAULT
resnet50=models.resnet50(weights=weights_ResNet50)
transform_ResNet50 = weights_ResNet50.transforms()
print(transform_ResNet50)

# MobileNet V2

In [ ]:
weight_mobilenet_v2=models.MobileNet_V2_Weights.DEFAULT
mobilenet_v2=models.mobilenet_v2(weights=weight_mobilenet_v2)
transform_mobilenet_v2 = weight_mobilenet_v2.transforms()
print(transform_mobilenet_v2)

## Auto transfom

In [ ]:
# Auto transfom
# auto_transforms = weight_mobilenet_v2.transforms()
# auto_transforms



## Model freeze and change output

In [ ]:
list_model=[resnet18,resnet50,mobilenet_v2]
dict_model = {
    "resnet18": resnet18,
    "resnet50": resnet50,
    "mobilenet_v2": mobilenet_v2
}


def initial_weights_model(model_name,model):
  #freeze
  for param in model.parameters():
    param.requires_grad = False

    #out_feature len(img)
    try:
      num_features = model.fc.in_features
      model.fc = nn.Linear(num_features, len(img))
    except:
      num_features = model.classifier[1].in_features
      model.classifier[1] = nn.Linear(num_features, len(img))

    #save initial weight
    torch.save(model.state_dict(), f'{model_name}_initial_weights.pth')

for model_name,model in dict_model.items():
  initial_weights_model(model_name,model)


# Criterion and optimizer

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(mobilenet_v2.parameters(), lr=0.0005)

# Training loop

In [ ]:
from tqdm.auto import tqdm
import torch

def train_step(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0

    for features, targets in tqdm(dataloader, desc="Training", leave=False):
        features, targets = features.to(device), targets.to(device)

        # Forward
        outputs = model(features)
        loss = criterion(outputs, targets)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate loss & accuracy
        total_loss += loss.item() * features.size(0)
        preds = torch.argmax(outputs, dim=1)
        total_correct += (preds == targets).sum().item()
        total_samples += features.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc


def test_step(model, dataloader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0, 0, 0

    with torch.no_grad():
        for features, targets in tqdm(dataloader, desc="Validating", leave=False):
            features, targets = features.to(device), targets.to(device)

            outputs = model(features)
            loss = criterion(outputs, targets)

            total_loss += loss.item() * features.size(0)
            preds = torch.argmax(outputs, dim=1)
            total_correct += (preds == targets).sum().item()
            total_samples += features.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc

In [ ]:
from os import write
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter()

def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs,writer):
    results = {"list_train_loss": [],
            "list_train_acc": [],
            "list_test_loss": [],
            "list_test_acc": []}
    for epoch in range(1, epochs + 1):
        print(f"\nEpoch [{epoch}/{epochs}]")

        train_loss, train_acc = train_step(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = test_step(model, test_loader, criterion, device)

        results['list_train_loss'].append(train_loss)
        results['list_train_acc'].append(train_acc)


        results['list_test_loss'].append(val_loss)
        results['list_test_acc'].append(val_acc)


        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Test   Loss: {val_loss:.4f} | Test   Acc: {val_acc:.4f}")
        print("=" * 60)

        if write:

          writer.add_scalars(main_tag="Loss",
                            tag_scalar_dict={"train_loss": train_loss,
                                              "test_loss": val_loss},
                            global_step=epoch)

          # Add accuracy results to SummaryWriter
          writer.add_scalars(main_tag="Accuracy",
                            tag_scalar_dict={"train_acc": train_acc,
                                              "test_acc": val_acc},
                            global_step=epoch)

          # Close the writer
          writer.close()
        else:
          pass
    return results




In [ ]:
# result=train_model(mobilenet_v2, train_loader, test_loader, criterion, optimizer, device, epochs=5)

In [ ]:
def create_writer(experiment_name: str,
                  model_name: str,
                  extra: str=None):
    """Creates a torch.utils.tensorboard.writer.SummaryWriter() instance saving to a specific log_dir.

    log_dir is a combination of runs/timestamp/experiment_name/model_name/extra.

    Where timestamp is the current date in YYYY-MM-DD format.

    Args:
        experiment_name (str): Name of experiment.
        model_name (str): Name of model.
        extra (str, optional): Anything extra to add to the directory. Defaults to None.

    Returns:
        torch.utils.tensorboard.writer.SummaryWriter(): Instance of a writer saving to log_dir.

    Example usage:
        # Create a writer saving to "runs/2022-06-04/data_10_percent/effnetb2/5_epochs/"
        writer = create_writer(experiment_name="data_10_percent",
                               model_name="effnetb2",
                               extra="5_epochs")
        # The above is the same as:
        writer = SummaryWriter(log_dir="runs/2022-06-04/data_10_percent/effnetb2/5_epochs/")
    """
    from datetime import datetime
    import os

    # Get timestamp of current date (all experiments on certain day live in same folder)
    timestamp = datetime.now().strftime("%Y-%m-%d") # returns current date in YYYY-MM-DD format

    if extra:
        # Create log directory path
        log_dir = os.path.join("experiment", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("experiment", timestamp, experiment_name, model_name)

    print(f"[INFO] Created SummaryWriter, saving to: {log_dir}...")
    return SummaryWriter(log_dir=log_dir)

In [ ]:
num_epochs=[3,5]
experiment_number=0
for epochs in num_epochs:
  for model_name,model in dict_model.items():
    experiment_number += 1
    print(f"[INFO] Experiment number: {experiment_number}")
    print(f"[INFO] Model: {model_name}")
    print(f"[INFO] Number of epochs: {epochs}")

    optimizer = optim.AdamW(model.parameters(), lr=0.0005) #optimizer
    model.load_state_dict(torch.load(f'{model_name}_initial_weights.pth'))
    train_model(model,
                train_loader,
                test_loader,
                criterion,
                optimizer,
                device,
                epochs=epochs,
                writer=create_writer(experiment_name="img classification",
                                     model_name=model_name,
                                     extra=str(epochs)))

    save_filepath = f"{model_name}_{epochs}_epochs.pth"
    torch.save(model.state_dict(), save_filepath)
    print("+"*50,'/n',"+"*50,'/n')


In [ ]:
# Example code to run in Jupyter or Google Colab Notebook
%load_ext tensorboard
%tensorboard --logdir runs